In [0]:
from pyspark.sql.functions import col, expr

print("🚀 Starting SILVER SHAPES processing...")

df = spark.read.format("delta").load(bronze_path)

print("🧹 Processing SHAPES...")

shapes = (
    df.select(
        col("shape_id"),
        expr("try_cast(shape_pt_lat as double)").alias("shape_pt_lat"),
        expr("try_cast(shape_pt_lon as double)").alias("shape_pt_lon"),
        col("shape_pt_sequence"),
        col("shape_dist_traveled")
    )
    .dropna(subset=["shape_id", "shape_pt_lat", "shape_pt_lon"])  # remove inválidos
    .dropDuplicates(["shape_id", "shape_pt_sequence"])
)

shapes.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_base_path}/shapes")

print("✅ SILVER SHAPES completed!")